In [18]:
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import sys
import torch
import os

import matplotlib.pyplot as plt
import hydra

import wandb

sys.path = ["..", *sys.path]

The jupyter_black extension is already loaded. To reload it, use:
  %reload_ext jupyter_black
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
def flatten_dict(d, parent_key="", sep="/"):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)


def prepare_data(data):
    flattened_data = [flatten_dict(item) for item in data]
    return pd.DataFrame(flattened_data)

In [7]:
api = wandb.Api(timeout=30)

# Project is specified by <entity/project-name>
runs = api.runs(
    "openproblems-comp/fast-tbg",
    filters={
        "$and": [
            {
                "created_at": {"$gt": "2025-01-26T00:00:00"},
                "tags": {"$in": ["plot_me"]},
                #'group': {'$in': ['5_vars']},
                # "config.data.n_particles": {"$eq": 22},
                #'config.model': {'$eq': model},
                #'config.lr': {'$lt': 1.01 * lr, '$gt': 0.99 * lr},
            }
        ]
    },
)

summary_list, config_list, name_list, tag_list = [], [], [], []
for run in runs:
    tag_list.append(run.tags)
    # .summary contains the output keys/values for metrics like accuracy.
    #  We call ._json_dict to omit large files
    summary_list.append(run.summary._json_dict)
    # .config contains the hyperparameters.
    #  We remove special values that start with _.
    config_list.append({k: v for k, v in run.config.items() if not k.startswith("_")})
    # .name is the human-readable name of the run.
    name_list.append(run.name)
df_summary = prepare_data(summary_list)
df_config = prepare_data(config_list)
tag_list = [str(t) for t in tag_list]
df = pd.concat(
    [
        pd.DataFrame(name_list, columns=["name"]),
        pd.DataFrame(tag_list, columns=["Tags"]),
        df_summary,
        df_config,
    ],
    axis=1,
)

In [8]:
df

,name,Tags,_runtime,_step,_timestamp,_wandb/runtime,epoch,test/1-Wasserstein,test/2-Wasserstein,test/Mean_L1,...,data/filename,data/pdb_filename,trainer/strategy,trainer/num_nodes,trainer/sync_batchnorm,model/sigma,model/scheduler/step_size,trainer/num_sanity_val_steps,data/data_url,model/net/pdb_filename
0,proud-lion-3268,"['al', 'cnf', 'eval', 'plot_me', 'v9']",7787.200204,4,1.737968e+09,7787,0,0.658858,0.669871,0.002496,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,prime-waterfall-3273,"['al', 'cnf', 'eval', 'plot_me', 'v9']",22455.632923,17,1.737995e+09,22455,0,0.990239,1.021321,0.001779,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,worthy-cosmos-3414,"['al', 'cnf', 'eval', 'plot_me', 'v9']",15075.665428,25,1.738024e+09,15075,0,1.383969,1.452077,0.004043,...,AceAlaAlaAlaNme_300K.npy,AceAlaAlaAlaNme_300K.pdb,ddp,1.0,True,NaN,NaN,NaN,NaN,NaN
3,winter-water-4405,"['dev', 'plot_me']",26766.403887,15854,1.738197e+09,26766,1000,1.141882,1.151315,0.002063,...,NaN,NaN,NaN,NaN,NaN,0.01,500.0,1.0,NaN,NaN
4,effortless-pine-4406,"['dev', 'plot_me']",41786.142418,15950,1.738212e+09,41786,1000,1.613594,1.628095,0.003204,...,AceAlaAlaAlaNme_300K.npy,AceAlaAlaAlaNme_300K.pdb,NaN,NaN,NaN,0.01,500.0,1.0,NaN,NaN
5,lyric-forest-4732,"['al', 'cnf', 'eval', 'plot_me', 'v9']",4248.781922,41,1.738242e+09,4248,0,2.389132,2.428334,0.008114,...,AAAAAA.npy,AAAAAA.pdb,ddp,1.0,True,NaN,NaN,NaN,https://osf.io/download/6x4wk/?view_only=af935...,AAAAAA.pdb


In [20]:
# df[[a for a in df.columns if "data/n" in a]]

In [41]:
def plot_samples(row, clip=0.002, overwrite=False):
    model = "Unknown"
    if "matching" in row["model/_target_"]:
        if row["model/sigma"] == 0.01:
            model = "ecnf"
        else:
            model = "ecnf++"
    else:
        model = "SBG"
    data_dir = row["data/data_dir"]
    n_particles = row["data/n_particles"]
    save_path = f"figs/density_v2/{n_particles}_{model}_density.png"
    rama_save_path = f"figs/rama_v2/{n_particles}_{model}_rama.png"
    if os.path.exists(save_path) and not overwrite:
        print(f"Skipping {save_path}")
        return
    if "bose" in data_dir:
        data_dir = "../data/alanine"
    sample_file = torch.load(f"{row['paths/output_dir']}/test_samples.pt", weights_only=True)
    samples = sample_file["samples"]
    logp = sample_file["log_p"]
    cfg = {
        "_target_": row["data/_target_"],
        "data_dir": data_dir,
        "batch_size": 512,  # Needs to be divisible by the number of devices (e.g., if in a distributed setup)
        "num_workers": 0,
        "n_particles": int(row["data/n_particles"]),
        "n_dimensions": row["data/n_dimensions"],
        "dim": int(row["data/dim"]),
    }
    if n_particles > 22 and row["data/filename"] == row["data/filename"]:
        cfg["filename"] = row["data/filename"]
        cfg["pdb_filename"] = row["data/pdb_filename"]
    datamodule = hydra.utils.instantiate(cfg)
    datamodule.setup()
    min_energy = -50
    max_energy = 20
    ylim = (0, 0.1)
    if n_particles == 63:
        min_energy = -130
        max_energy = -40
        ylim = (0, 0.1)
    if n_particles == 53:
        min_energy = -150
        max_energy = -60
        ylim = (0, 0.1)
    if n_particles == 42:
        min_energy = -20
        max_energy = 80
        ylim = (0, 0.1)
    if n_particles == 33:
        min_energy = -200
        max_energy = -100
        ylim = (0, 0.1)
    # max_energy = 100
    datamodule.plot_nice_samples(
        samples,
        log_p_samples=logp,
        min_energy=min_energy,
        max_energy=max_energy,
        clip_energy=max_energy,
        ylim=ylim,
    )
    print(f"saving plot to {save_path}")
    plt.savefig(save_path, dpi=300)
    plt.close()

    # ramaplot
    from matplotlib.colors import LogNorm

    x_pred = datamodule.get_phi_psi_vectors(datamodule.unnormalize(samples).cpu())
    phis_data, psis_data = x_pred[:, : x_pred.shape[1] // 2], x_pred[:, x_pred.shape[1] // 2 :]
    print(x_pred.shape, phis_data.shape, psis_data.shape)
    n_angles = phis_data.shape[1]
    print(n_angles)
    fig, axs = plt.subplots(1, n_angles, figsize=(n_angles * 3, 3), constrained_layout=True)
    plot_range = [-np.pi, np.pi]
    for i in range(phis_data.shape[1]):
        if phis_data.shape[1] == 1:
            ax = axs
        else:
            ax = axs[i]
        h1, x_bins1, y_bins1, im1 = ax.hist2d(
            phis_data[:, i],
            psis_data[:, i],
            100,
            norm=LogNorm(),
            range=[plot_range, plot_range],
            rasterized=True,
        )
        ax.set_xlabel(rf"$\varphi_{i+1}$", labelpad=-15, fontsize=15)  # , fontsize=45)
        ax.set_ylabel(rf"$\psi_{i+1}$", labelpad=-15, fontsize=15)  # , fontsize=45)
        ax.set_xticks([-np.pi / 2, np.pi / 2])
        ax.set_xticklabels([r"$-\frac{\pi}{2}$", r"$\frac{\pi}{2}$"], fontsize=12)
        ax.set_yticks([-np.pi / 2, np.pi / 2])
        ax.set_yticklabels([r"$-\frac{\pi}{2}$", r"$\frac{\pi}{2}$"], fontsize=12)
    print(f"Saving plot {rama_save_path}")
    plt.savefig(rama_save_path, dpi=300)
    plt.close()


for i, row in df[::-1].iterrows():
    plot_samples(row, overwrite=True)

saving plot to figs/density_v2/63_ecnf++_density.png
Saving plot figs/rama_v2/63_ecnf++_rama.png
saving plot to figs/density_v2/42_ecnf_density.png
Saving plot figs/rama_v2/42_ecnf_rama.png
saving plot to figs/density_v2/33_ecnf_density.png
Saving plot figs/rama_v2/33_ecnf_rama.png
saving plot to figs/density_v2/42_ecnf++_density.png
Saving plot figs/rama_v2/42_ecnf++_rama.png
saving plot to figs/density_v2/33_ecnf++_density.png
Saving plot figs/rama_v2/33_ecnf++_rama.png
Using downloaded and verified file: /tmp/A.pdb
saving plot to figs/density_v2/22_ecnf++_density.png
Saving plot figs/rama_v2/22_ecnf++_rama.png


In [138]:
# npz_file = np.load(f"/result_data/ad2_tbg_full.npz")
def plot_ecnf():
    """
    Plotting for results in format of TBG code.
    """
    from bgflow import MeanFreeNormalDistribution

    npz_file = np.load(f"../old/result_data/Flow-Matching-AD2-amber-weighted-encoding.npz")

    latent_np = npz_file["latent_np"]
    samples_np = npz_file["samples_np"]
    dlogp_np = npz_file["dlogp_np"]
    dim = 66
    n_particles = 22
    prior = MeanFreeNormalDistribution(dim, n_particles, two_event_dims=False)
    logp = -prior.energy(torch.from_numpy(latent_np)) - torch.from_numpy(dlogp_np).reshape(-1, 1)
    samples = torch.from_numpy(samples_np)

    cfg = {
        "_target_": "src.data.aldp_datamodule.ALDPDataModule",
        "data_dir": "../data/AD2/",
        "batch_size": 512,  # Needs to be divisible by the number of devices (e.g., if in a distributed setup)
        "num_workers": 0,
        "n_particles": 22,
        "n_dimensions": 3,
        "dim": 66,
    }
    datamodule = hydra.utils.instantiate(cfg)
    datamodule.setup()
    samples = samples / 10 / datamodule.std
    datamodule.plot_nice_samples(samples, log_p_samples=logp, min_energy=-50, max_energy=25)
    plt.savefig(f"al2_ecnf_density.png", dpi=300)
    plt.close()


plot_ecnf()

Using downloaded and verified file: /tmp/A.pdb
